# ARC All-in-One Notebook (Kaggle GPU Ready)
Merged pipeline from notebooks 01-06: data, model, training, evaluation, predictions, and ablations.

ARC evaluation rule followed:
- Development uses only `data/training`.
- `data/evaluation` is used only for final inference.
- Evaluation test outputs are stripped before prediction.

## Kaggle Run Steps
1. In Kaggle Notebook settings, set Accelerator to **GPU**.
2. Add this project folder as a Kaggle Dataset (or copy files into `/kaggle/working`).
3. Run cells top to bottom.
4. Set `TRAIN_FROM_SCRATCH = True` only when you want to train in Kaggle; otherwise upload checkpoint and keep it `False`.

In [ ]:
import os
import sys
from pathlib import Path

# Helps reduce CUDA memory fragmentation on long Kaggle runs.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path('/kaggle/working'), Path('/kaggle/input')]
    for base in candidates:
        if not base.exists():
            continue

        level_one = [base]
        try:
            level_one.extend([p for p in base.iterdir() if p.is_dir()])
        except Exception:
            pass

        level_two = []
        for p in level_one[1:]:
            try:
                level_two.extend([q for q in p.iterdir() if q.is_dir()])
            except Exception:
                continue

        for p in level_one + level_two:
            if (p / 'arc_system.py').exists():
                return p

    raise FileNotFoundError('Could not find project root containing arc_system.py')

def find_data_root(project_root: Path) -> Path:
    if (project_root / 'data').exists():
        return project_root / 'data'

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for ds in kaggle_input.iterdir():
            if not ds.is_dir():
                continue
            if (ds / 'data').exists():
                return ds / 'data'
            if (ds / 'raw').exists() and (ds / 'processed').exists():
                return ds

    raise FileNotFoundError('Could not find a data directory. Expected <root>/data or /kaggle/input/<dataset>/data')

def choose_output_dir(project_root: Path) -> Path:
    candidates = [Path('/kaggle/working/outputs'), Path('/kaggle/output'), project_root / 'outputs']
    for out_dir in candidates:
        try:
            out_dir.mkdir(parents=True, exist_ok=True)
            probe = out_dir / '.write_probe'
            probe.write_text('ok', encoding='utf-8')
            probe.unlink()
            return out_dir
        except Exception:
            continue
    raise PermissionError('No writable output directory found.')

IS_KAGGLE = Path('/kaggle').exists()
ROOT = find_project_root()
DATA_ROOT = find_data_root(ROOT)    
OUTPUT_DIR = choose_output_dir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)
print('Data root:', DATA_ROOT)
print('Output dir:', OUTPUT_DIR)
print('Running on Kaggle:', IS_KAGGLE)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from arc_system import (
    ARCExample,
    ARCModelConfig,
    TrainingConfig,
    AugmentationConfig,
    ARCTransformer,
    ARCTokenDataset,
    build_loaders_from_raw,
    count_parameters,
    create_dataloader,
    default_device,
    download_arc_dataset,
    evaluate_model,
    load_arc_examples,
    load_model_from_checkpoint,
    model_under_param_budget,
    plot_prediction,
    run_ablation_suite,
    save_json,
    set_seed,
    split_train_val,
    train_model,
)

In [ ]:
RAW_DIR = DATA_ROOT / 'raw'
PROCESSED_DIR = DATA_ROOT / 'processed'
(OUTPUT_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

if not (RAW_DIR / 'ARC-AGI-master').exists():
    if IS_KAGGLE and str(RAW_DIR).startswith('/kaggle/input'):
        # Kaggle input is read-only: download to writable working area if dataset is missing raw files.
        RAW_DIR = OUTPUT_DIR / 'data' / 'raw'
        RAW_DIR.mkdir(parents=True, exist_ok=True)
    download_arc_dataset(RAW_DIR)

set_seed(42)
device = default_device()
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('Device selected:', device)

In [ ]:
# Options: full | kaggle_safe | kaggle_ultra_safe
MEMORY_PROFILE = 'kaggle_safe' if IS_KAGGLE else 'full'

if MEMORY_PROFILE == 'kaggle_ultra_safe':
    model_cfg = ARCModelConfig(
        d_model=192,
        nhead=6,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=768,
        dropout=0.1,
        pos_encoding='learned',
    )
    train_cfg = TrainingConfig(
        seed=42,
        epochs=20,
        lr=2e-4,
        grad_accum_steps=8,
        mixed_precision='fp16',
        max_tokens_per_batch=2000,
        max_batch_size=2,
        val_ratio=0.15,
        checkpoint_name='arc_best.pt',
        refinement_steps_eval=1,
    )
elif MEMORY_PROFILE == 'kaggle_safe':
    model_cfg = ARCModelConfig(
        d_model=256,
        nhead=8,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=1024,
        dropout=0.1,
        pos_encoding='learned',
    )
    train_cfg = TrainingConfig(
        seed=42,
        epochs=20,
        lr=2e-4,
        grad_accum_steps=4,
        mixed_precision='fp16',
        max_tokens_per_batch=3500,
        max_batch_size=4,
        val_ratio=0.15,
        checkpoint_name='arc_best.pt',
        refinement_steps_eval=1,
    )
else:
    model_cfg = ARCModelConfig(
        d_model=384,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=1536,
        dropout=0.1,
        pos_encoding='learned',
    )
    train_cfg = TrainingConfig(
        seed=42,
        epochs=20,
        lr=2e-4,
        grad_accum_steps=2,
        mixed_precision='fp16',
        max_tokens_per_batch=14000,
        max_batch_size=16,
        val_ratio=0.15,
        checkpoint_name='arc_best.pt',
        refinement_steps_eval=2,
    )

aug_cfg = AugmentationConfig(
    enabled=True,
    rotate=True,
    flip_horizontal=True,
    flip_vertical=True,
    color_permutation_prob=0.1,
    keep_zero_fixed=True,
)

model = ARCTransformer(model_cfg)
params = count_parameters(model)
print('Memory profile:', MEMORY_PROFILE)
print('Parameter count:', params)
print('Within 50M budget:', model_under_param_budget(model, 50_000_000))
print('Batch budget:', train_cfg.max_tokens_per_batch, train_cfg.max_batch_size, train_cfg.grad_accum_steps)

In [ ]:
variant_cfg = ARCModelConfig(pos_encoding='sinusoidal')
variant_model = ARCTransformer(variant_cfg)
print('Sinusoidal variant params:', count_parameters(variant_model))
rotary_cfg = ARCModelConfig(pos_encoding='learned', use_rotary=True)
rotary_model = ARCTransformer(rotary_cfg)
print('Rotary variant params:', count_parameters(rotary_model))

In [ ]:
train_loader, val_loader, train_split, val_split = build_loaders_from_raw(
    raw_dir=RAW_DIR,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    augment_cfg=aug_cfg,
)

print('Train examples:', len(train_split))
print('Val examples:', len(val_split))
print('Train batches:', len(train_loader))
print('Val batches:', len(val_loader))

batch = next(iter(train_loader))
print('Batch token tensor:', batch['colors'].shape)
print('Batch attention mask:', batch['attention_mask'].shape)
print('Batch targets:', batch['target_grid'].shape)
print('Example task IDs:', batch['task_ids'][:3])

In [ ]:
TRAIN_FROM_SCRATCH = False
CKPT = OUTPUT_DIR / 'checkpoints' / train_cfg.checkpoint_name

if TRAIN_FROM_SCRATCH or not CKPT.exists():
    try:
        training_result = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            train_cfg=train_cfg,
            output_dir=OUTPUT_DIR,
            device=device,
        )
        print(training_result)
    except torch.OutOfMemoryError:
        if not torch.cuda.is_available():
            raise
        print('CUDA OOM hit. Retrying once with ultra-safe settings...')
        import gc
        gc.collect()
        torch.cuda.empty_cache()

        train_cfg.max_tokens_per_batch = max(1000, train_cfg.max_tokens_per_batch // 2)
        train_cfg.max_batch_size = max(1, train_cfg.max_batch_size // 2)
        train_cfg.grad_accum_steps = max(1, train_cfg.grad_accum_steps * 2)
        train_cfg.refinement_steps_eval = 1

        train_loader, val_loader, train_split, val_split = build_loaders_from_raw(
            raw_dir=RAW_DIR,
            model_cfg=model_cfg,
            train_cfg=train_cfg,
            augment_cfg=aug_cfg,
        )

        print(
            'Retry budget:',
            train_cfg.max_tokens_per_batch,
            train_cfg.max_batch_size,
            train_cfg.grad_accum_steps,
        )
        training_result = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            train_cfg=train_cfg,
            output_dir=OUTPUT_DIR,
            device=device,
        )
        print(training_result)
else:
    model = load_model_from_checkpoint(CKPT, device=device)
    print('Loaded checkpoint:', CKPT)

In [ ]:
# Validation uses only the official training split.
all_training = load_arc_examples(RAW_DIR, split='training')
_, val_examples = split_train_val(all_training, val_ratio=train_cfg.val_ratio, seed=train_cfg.seed)
val_ds = ARCTokenDataset(
    examples=val_examples,
    model_cfg=model.config,
    training=False,
    augment_config=None,
    seed=42,
)
val_loader = create_dataloader(
    dataset=val_ds,
    max_tokens_per_batch=train_cfg.max_tokens_per_batch,
    max_batch_size=train_cfg.max_batch_size,
    shuffle=False,
    num_workers=0,
)
val_result = evaluate_model(model, val_loader, device=device, refinement_steps=2)
print('Validation exact-match:', val_result['overall_exact_match'])

In [ ]:
sample = val_examples[0]
sample_pred = val_result['predictions'][sample.task_id]
plot_prediction(sample, sample_pred)

## Optional Ablations
Set `RUN_ABLATIONS = True` only when you intentionally want to run multiple experiments.

In [ ]:
RUN_ABLATIONS = False

if RUN_ABLATIONS:
    ABLATION_DIR = OUTPUT_DIR / 'ablations'
    base_model_cfg = ARCModelConfig(
        d_model=320,
        nhead=8,
        num_encoder_layers=5,
        num_decoder_layers=5,
        dim_feedforward=1280,
        pos_encoding='learned',
    )
    base_train_cfg = TrainingConfig(
        epochs=8,
        lr=2e-4,
        grad_accum_steps=2,
        mixed_precision='fp16',
        max_tokens_per_batch=12000,
        max_batch_size=12,
        checkpoint_name='ablation_best.pt',
    )
    base_aug_cfg = AugmentationConfig(
        enabled=True,
        rotate=True,
        flip_horizontal=True,
        flip_vertical=True,
        color_permutation_prob=0.1,
    )
    variants = [
        {
            'name': 'baseline_learned_pos_aug',
            'model': {'pos_encoding': 'learned'},
            'augment': {'enabled': True},
        },
        {
            'name': 'sinusoidal_pos_aug',
            'model': {'pos_encoding': 'sinusoidal'},
            'augment': {'enabled': True},
        },
        {
            'name': 'learned_pos_no_aug',
            'model': {'pos_encoding': 'learned'},
            'augment': {'enabled': False},
        },
        {
            'name': 'learned_pos_rotary_aug',
            'model': {'pos_encoding': 'learned', 'use_rotary': True},
            'augment': {'enabled': True},
        },
    ]
    ablation_df = run_ablation_suite(
        raw_dir=RAW_DIR,
        output_dir=ABLATION_DIR,
        model_variants=variants,
        base_model_cfg=base_model_cfg,
        base_train_cfg=base_train_cfg,
        base_aug_cfg=base_aug_cfg,
        device=device,
    )
    ablation_df
else:
    print('Skipping ablations')

## Final ARC Evaluation Inference (Inputs Only)
This section uses `data/evaluation` only to produce predictions. It explicitly removes test outputs so no labels are used.

In [ ]:
eval_examples_raw = load_arc_examples(RAW_DIR, split='evaluation')
eval_examples = [
    ARCExample(
        task_id=ex.task_id,
        demo_pairs=ex.demo_pairs,
        test_input=ex.test_input,
        test_output=None,
    )
    for ex in eval_examples_raw
]
assert all(ex.test_output is None for ex in eval_examples), 'Evaluation outputs must not be used.'

eval_ds = ARCTokenDataset(
    examples=eval_examples,
    model_cfg=model.config,
    training=False,
    augment_config=None,
    seed=42,
)
eval_loader = create_dataloader(
    dataset=eval_ds,
    max_tokens_per_batch=train_cfg.max_tokens_per_batch,
    max_batch_size=train_cfg.max_batch_size,
    shuffle=False,
    num_workers=0,
)
print('Evaluation examples (input-only):', len(eval_examples))

In [ ]:
# evaluate_model is used only as a prediction wrapper here; with no targets present it does not score.
eval_result = evaluate_model(model, eval_loader, device=device, refinement_steps=2)
eval_predictions = eval_result['predictions']
print('Evaluation predictions generated:', len(eval_predictions))

In [ ]:
pred_path = OUTPUT_DIR / 'logs' / 'evaluation_predictions_inputs_only.json'
save_json(pred_path, {k: v.tolist() for k, v in eval_predictions.items()})
print('Saved predictions to:', pred_path)